# Storyboard narration generator (recipe @2: Indic Parler-TTS, Mary + Divya)
Makes the narration MP3s from a plan file made on the owner's PC (`pnpm narration:export`).
Each file is spoken sentence by sentence exactly as in the approved test, then levelled to a -1 dB peak, then saved as MP3 (24 kHz, 48 kbps, mono).
Output: `My Drive/storyboard-narration/` with one folder per story and `manifest.jsonl`. It **resumes** after a disconnect: finished files are skipped.
The first run pins the exact model revision, Parler-TTS code commit and tokenizer revision in that Drive folder; every later session uses exactly those.

**Before Run all:** Runtime -> Change runtime type -> **T4 GPU** (step 1 stops if the runtime has no GPU). Colab secret `HF_TOKEN` (key icon, Notebook access on), as in the test.
Step 2 asks to connect Google Drive. Step 5 asks for **one** plan file: `storyboard\migration-input\narration\parler-plan.json` (all files), or for a remake the `parler-plan.redo-<stamp>.json` that the export printed. If an earlier redo file was never run here, upload `parler-plan.json` instead (finished files are skipped). A plan older than one already used here is refused.
When step 7 prints `all done`, download the zip from step 8, let step 9 finish syncing Drive, extract the zip, and on the PC run `pnpm narration:ingest <extracted folder>`.


In [ ]:
# 1. PyTorch only (Colab's TensorFlow clashes with the protobuf Parler-TTS installs); GPU check
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TRANSFORMERS_NO_TF'] = '1'
import torch
if not torch.cuda.is_available():
    # Files made without a GPU came out different (much shorter Hindi pauses, measured 2026-11-06): never make them.
    raise RuntimeError('No GPU in this runtime: Runtime -> Change runtime type -> T4 GPU, then Run all again. '
                       'If Colab says the GPU usage limit is reached, stop and try again later; '
                       'do not choose "Connect without GPU".')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Google Drive: results (and the pinned versions) survive disconnects
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/storyboard-narration'
os.makedirs(os.path.join(OUT, 'plans'), exist_ok=True)
print('output folder:', OUT)


In [ ]:
# 3. Install Parler-TTS at the code commit pinned by the first run (this Colab runtime only).
#    Red pip warnings about other packages are expected.
assert os.path.ismount('/content/drive'), 'Google Drive is not connected: run step 2 again'
import subprocess
PIN_FILE = os.path.join(OUT, 'parler-tts-commit.txt')
if os.path.exists(PIN_FILE):
    PARLER_COMMIT = open(PIN_FILE, encoding='utf-8').read().strip()
    print('using the Parler-TTS commit pinned by the first run:', PARLER_COMMIT)
else:
    PARLER_COMMIT = subprocess.run(['git', 'ls-remote', 'https://github.com/huggingface/parler-tts.git', 'HEAD'],
                                   check=True, capture_output=True, text=True).stdout.split()[0]
    with open(PIN_FILE, 'w', encoding='utf-8') as f:
        f.write(PARLER_COMMIT + '\n')
    print('pinned the Parler-TTS commit for all later runs:', PARLER_COMMIT)
!pip -q install "git+https://github.com/huggingface/parler-tts.git@{PARLER_COMMIT}" soundfile


In [ ]:
# 4. Hugging Face sign-in (gated model). Waits for a token: Colab secret HF_TOKEN, else a hidden prompt.
from huggingface_hub import login, whoami
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass('Paste your Hugging Face Read token (hidden), then press Enter: ').strip()
login(token=token, add_to_git_credential=False)
print('Signed in to Hugging Face as:', whoami()['name'])


In [ ]:
# 5. Upload ONE plan file from the PC: parler-plan.json (all files) or a parler-plan.redo-<stamp>.json
import json
from google.colab import files
assert os.path.ismount('/content/drive'), 'Google Drive is not connected: run step 2 again'
PLANS_DIR = os.path.join(OUT, 'plans')
# Newest plan already used here for each file: an older plan must not overwrite a newer take.
used = {}
for fn in os.listdir(PLANS_DIR):
    if not fn.endswith('.json'):
        continue
    try:
        old = json.load(open(os.path.join(PLANS_DIR, fn), encoding='utf-8'))
    except Exception:
        continue
    for e in old.get('entries', []):
        k = (e['storyId'], e['nodeId'], e['locale'])
        if k not in used or old['createdAt'] > used[k][0]:
            used[k] = (old['createdAt'], e['narratedTextSha256'])
uploaded = files.upload()
plans = []
for name, data in uploaded.items():
    plan = json.loads(data.decode('utf-8'))
    assert plan['recipe']['version'] == 'narration-recipe@2', 'unexpected recipe version'
    assert 'levelPeakDbfs' in plan['recipe'], 'old plan file: run pnpm narration:export again on the PC'
    for e in plan['entries']:
        k = (e['storyId'], e['nodeId'], e['locale'])
        if k in used and used[k][0] > plan['createdAt'] and used[k][1] != e['narratedTextSha256']:
            raise RuntimeError(f'{name} is older than a plan already used here for {k} ({used[k][0]}): '
                               'upload the newest plan file from the PC')
    plans.append((name, data, plan))
assert plans, 'no plan file uploaded'
assert all(p['recipe'] == plans[0][2]['recipe'] for _, _, p in plans), 'the uploaded plans use different recipes'
ENTRIES, seen = [], set()
for _, _, p in plans:
    for e in p['entries']:
        k = (e['storyId'], e['nodeId'], e['locale'])
        assert k not in seen, f'{k} is in more than one uploaded plan: upload only ONE plan file'
        seen.add(k)
        ENTRIES.append(e)
for name, data, p in plans:  # keep a copy of each accepted plan, named by its creation time
    with open(os.path.join(PLANS_DIR, p['createdAt'].replace(':', '').replace('-', '') + '__' + name), 'wb') as f:
        f.write(data)
RECIPE = plans[0][2]['recipe']
print(len(ENTRIES), 'files in the uploaded plan; recipe', RECIPE['version'])


In [ ]:
# 6. Load the model and the description tokenizer at the revisions pinned by the first run
assert os.path.ismount('/content/drive'), 'Google Drive is not connected: run step 2 again'
import time, hashlib, math, shutil, datetime, numpy as np, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, set_seed
from huggingface_hub import model_info
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
MODEL_ID = RECIPE['model']
REV_FILE = os.path.join(OUT, 'model-revision.txt')
if os.path.exists(REV_FILE):
    REVISION = open(REV_FILE, encoding='utf-8').read().strip()
    print('using the model revision pinned by the first run:', REVISION)
else:
    REVISION = model_info(MODEL_ID).sha
    with open(REV_FILE, 'w', encoding='utf-8') as f:
        f.write(REVISION + '\n')
    print('pinned the model revision for all later runs:', REVISION)
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID, revision=REVISION).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
TOK_ID = model.config.text_encoder._name_or_path
TOK_FILE = os.path.join(OUT, 'tokenizer-revision.txt')
if os.path.exists(TOK_FILE):
    saved_id, TOK_REVISION = open(TOK_FILE, encoding='utf-8').read().split()
    assert saved_id == TOK_ID, f'the description tokenizer changed: {saved_id} -> {TOK_ID}'
    print('using the tokenizer revision pinned by the first run:', TOK_ID, TOK_REVISION)
else:
    TOK_REVISION = model_info(TOK_ID).sha
    with open(TOK_FILE, 'w', encoding='utf-8') as f:
        f.write(f'{TOK_ID} {TOK_REVISION}\n')
    print('pinned the tokenizer revision for all later runs:', TOK_ID, TOK_REVISION)
description_tokenizer = AutoTokenizer.from_pretrained(TOK_ID, revision=TOK_REVISION)
SR = model.config.sampling_rate
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
# Longest audio one generate() call can return: max_length codec frames minus the codebook delay.
MAX_LEN = model.generation_config.max_length
FRAME_RATE = getattr(model.audio_encoder.config, 'frame_rate', None) or SR / 512
N_CODEBOOKS = getattr(model.decoder.config, 'num_codebooks', 9)
CAP_S = (MAX_LEN - N_CODEBOOKS) / FRAME_RATE if MAX_LEN else None
print('model', MODEL_ID, '| sampling rate', SR, '| max_length', MAX_LEN, '| frame rate', FRAME_RATE,
      '| longest sentence in one go: about', 'unknown (cut-off check off)' if CAP_S is None else f'{CAP_S:.1f} s')


In [ ]:
# 7. Make every planned file that is not done yet (same fingerprint AND an MP3 matching its recorded hash)
assert os.path.ismount('/content/drive'), 'Google Drive is not connected: run step 2 again'
if device == 'cpu':
    raise RuntimeError('No GPU: nothing was made. Runtime -> Change runtime type -> T4 GPU, then Run all again.')
MANIFEST = os.path.join(OUT, 'manifest.jsonl')
TMP_WAV, TMP_MP3 = '/content/tmp.wav', '/content/tmp.mp3'

def fsync_quietly(f):
    f.flush()
    try:
        os.fsync(f.fileno())
    except OSError:
        pass

# Read the manifest; repair it if an earlier session died in the middle of writing a line.
done, good, bad = {}, [], 0
if os.path.exists(MANIFEST):
    raw = open(MANIFEST, encoding='utf-8').read()
    for line in raw.splitlines():
        if not line.strip():
            continue
        try:
            x = json.loads(line)
        except json.JSONDecodeError:
            bad += 1
            continue
        good.append(line + '\n')
        done[(x['storyId'], x['nodeId'], x['locale'])] = (x['narratedTextSha256'], x.get('mp3Sha256'))
    if bad or (raw and not raw.endswith('\n')):
        backup = MANIFEST + '.bak-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        shutil.copy(MANIFEST, backup)
        with open(MANIFEST, 'w', encoding='utf-8') as f:
            f.writelines(good)
            fsync_quietly(f)
        print(f'repaired manifest.jsonl ({bad} incomplete line(s) removed); backup: {backup}')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def is_done(e):
    d = done.get((e['storyId'], e['nodeId'], e['locale']))
    dst = os.path.join(OUT, e['file'])
    return (d is not None and d[0] == e['narratedTextSha256'] and d[1] is not None
            and os.path.exists(dst) and file_sha256(dst) == d[1])

def say(text, description, seed):
    set_seed(seed)
    d = description_tokenizer(description, return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                       prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    return np.atleast_1d(g.cpu().numpy().squeeze()).astype(np.float32)

FRAME_S, LOUD_RMS = 0.02, 0.02  # 20 ms frames; loud = RMS >= 0.02 (about -34 dBFS) after levelling

def loud_frames(a):
    # Judged on 20 ms frames, not single samples: Mary's background hiss crosses any per-sample level.
    f = int(SR * FRAME_S)
    n = len(a) // f
    if n == 0:
        return np.array([], dtype=np.int64)
    rms = np.sqrt(np.mean(a[:n * f].reshape(n, f) ** 2, axis=1))
    return np.flatnonzero(rms >= LOUD_RMS)

def inner_silence_s(a):
    # Longest run of quiet frames between the first and last loud frame of ONE sentence: the voice's own
    # lead-in and lead-out (the long pauses the owner approved) are not counted.
    loud = loud_frames(a)
    return float((np.diff(loud) - 1).max()) * FRAME_S if loud.size >= 2 else 0.0

def loud_span_s(a):
    # First to last loud frame: (almost) 0 means the sentence has no audible sound.
    loud = loud_frames(a)
    return float(loud[-1] - loud[0] + 1) * FRAME_S if loud.size else 0.0

print('checking files already made ...')
todo = [e for e in ENTRIES if not is_done(e)]
print(f'{len(todo)} to make, {len(ENTRIES) - len(todo)} already done')
t_start, warnings = time.time(), []
for i, e in enumerate(todo, 1):
    t0 = time.time()
    # Exactly as the approved test: sentence, 0.25 s gap, ...; the last gap becomes 0.8 s before the question.
    segs = []
    for s in e['sentences']:
        segs += [(True, say(s, e['description'], e['seed'])),
                 (False, np.zeros(int(SR * e['sentenceGapS']), dtype=np.float32))]
    if e['question']:
        if segs:
            segs[-1] = (False, np.zeros(int(SR * e['questionGapS']), dtype=np.float32))
        segs.append((True, say(e['question'], e['description'], e['seed'])))
    speech = [a for is_speech, a in segs if is_speech]
    audio = np.concatenate([a for _, a in segs])
    near_cap = [k for k, a in enumerate(speech, 1) if CAP_S is not None and len(a) / SR >= CAP_S - 0.5]
    # Level to the recipe's peak (-1 dBFS): one plain volume change for the whole file.
    raw_peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    gain = 10 ** (e['levelPeakDbfs'] / 20) / raw_peak if raw_peak > 0 else 1.0
    audio = (audio * gain).astype(np.float32)
    longest = max((inner_silence_s(a * gain) for a in speech), default=0.0)
    quiet = [k for k, a in enumerate(speech, 1) if loud_span_s(a * gain) < 0.15]
    # Encode on the local disk, measure the decoded MP3, then copy to Drive under a temporary name and rename.
    sf.write(TMP_WAV, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', TMP_WAV, '-ac', '1', '-ar', '24000',
                    '-c:a', 'libmp3lame', '-b:a', '48k', TMP_MP3], check=True)
    decoded = subprocess.run(['ffmpeg', '-loglevel', 'error', '-i', TMP_MP3, '-f', 'f32le', '-ac', '1', '-'],
                             check=True, capture_output=True).stdout
    mp3_peak = float(np.max(np.abs(np.frombuffer(decoded, dtype=np.float32)))) if decoded else 0.0
    data = open(TMP_MP3, 'rb').read()
    dst = os.path.join(OUT, e['file'])
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst + '.part', 'wb') as f:
        f.write(data)
        fsync_quietly(f)
    os.replace(dst + '.part', dst)
    line = {'storyId': e['storyId'], 'nodeId': e['nodeId'], 'locale': e['locale'],
            'narratedTextSha256': e['narratedTextSha256'], 'seed': e['seed'],
            'mp3Sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data),
            'rawPeak': round(raw_peak, 5), 'gainDb': round(20 * math.log10(gain), 2),
            'mp3Peak': round(mp3_peak, 5), 'longestSilenceS': round(longest, 2),
            'nearCapParts': near_cap, 'quietParts': quiet, 'capS': None if CAP_S is None else round(CAP_S, 2),
            'modelRevision': REVISION, 'parlerCommit': PARLER_COMMIT, 'tokenizerRevision': TOK_REVISION,
            'gpu': GPU, 'generatedAt': datetime.datetime.now(datetime.timezone.utc).isoformat()}
    with open(MANIFEST, 'a', encoding='utf-8') as f:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')
        fsync_quietly(f)
    note = []
    if near_cap:
        note.append(f'CHECK: part {near_cap} reached the length limit')
    if quiet:
        note.append(f'CHECK: part {quiet} has no audible sound')
    if mp3_peak >= 0.999:
        note.append(f'CHECK: MP3 peak {mp3_peak:.3f}')
    if note:
        warnings.append(f"{e['file']}: " + '; '.join(note))
    took = time.time() - t0
    eta_min = (time.time() - t_start) / i * (len(todo) - i) / 60
    print(f"[{i}/{len(todo)}] {e['file']}: {len(audio) / SR:.1f} s audio in {took:.0f} s, gain {20 * math.log10(gain):+.1f} dB,"
          f" MP3 peak {mp3_peak:.3f}{' | ' + '; '.join(note) if note else ''} | about {eta_min:.0f} min left")
print('all done' + (f' - {len(warnings)} file(s) to check (the PC ingest will flag them too):' if warnings else ''))
for w in warnings:
    print('  ', w)


In [ ]:
# 8. Download everything as one zip (preferred over downloading the Drive folder)
zip_base = '/content/storyboard-narration'
if os.path.ismount('/content/drive'):
    shutil.make_archive(zip_base, 'zip', OUT)
elif os.path.exists(zip_base + '.zip'):
    print('Google Drive is not connected; downloading the zip made earlier in this session')
else:
    raise RuntimeError('Google Drive is not connected: run step 2 again')
files.download(zip_base + '.zip')


In [ ]:
# 9. Finish writing everything to Google Drive (wait for 'Drive synced' before closing the tab)
drive.flush_and_unmount()
print('Drive synced')
